# 🔧 Notebook 07: Preprocesamiento — Telco Customer Churn

**Objetivo:** Limpiar, codificar y escalar el dataset de Churn para prepararlo para modelado.

**Pasos:**
1. Limpieza (duplicados, tipos, nulos)
2. Ingeniería de características base
3. Codificación (One-Hot + Label Encoding)
4. Escalamiento (StandardScaler)
5. División train/test estratificada (preservar balance)
6. Guardado de artefactos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib, json, warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print('✅ Librerías importadas')

✅ Librerías importadas


## 1. Carga y Limpieza

In [2]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'📦 Shape original: {df.shape}')

# ── 1.1 CONVERTIR TotalCharges a numérico
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# ── 1.2 IMPUTAR nulos en TotalCharges (tenure=0 → TotalCharges=0)
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f'✅ TotalCharges convertida. Nulos restantes: {df["TotalCharges"].isnull().sum()}')

# ── 1.3 DUPLICADOS
n_dup = df.duplicated().sum()
print(f'🔍 Duplicados: {n_dup}')
if n_dup > 0:
    df = df.drop_duplicates()

# ── 1.4 ELIMINAR customerID (no predictivo)
df = df.drop(columns=['customerID'])
print(f'✅ customerID eliminado. Shape: {df.shape}')

# ── 1.5 NORMALIZAR texto
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print('\n📊 Valores únicos por columna categórica:')
for col in df.select_dtypes(include='object').columns:
    print(f'  {col}: {df[col].unique().tolist()}')

📦 Shape original: (7043, 21)
✅ TotalCharges convertida. Nulos restantes: 0
🔍 Duplicados: 0
✅ customerID eliminado. Shape: (7043, 20)

📊 Valores únicos por columna categórica:
  gender: ['Female', 'Male']
  Partner: ['Yes', 'No']
  Dependents: ['No', 'Yes']
  PhoneService: ['No', 'Yes']
  MultipleLines: ['No phone service', 'No', 'Yes']
  InternetService: ['DSL', 'Fiber optic', 'No']
  OnlineSecurity: ['No', 'Yes', 'No internet service']
  OnlineBackup: ['Yes', 'No', 'No internet service']
  DeviceProtection: ['No', 'Yes', 'No internet service']
  TechSupport: ['No', 'Yes', 'No internet service']
  StreamingTV: ['No', 'Yes', 'No internet service']
  StreamingMovies: ['No', 'Yes', 'No internet service']
  Contract: ['Month-to-month', 'One year', 'Two year']
  PaperlessBilling: ['Yes', 'No']
  PaymentMethod: ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']
  Churn: ['No', 'Yes']


## 2. Ingeniería de Características Base

In [3]:
# ─────────────────────────────────────────
# FEATURES DERIVADAS (detectadas en EDA)
# ─────────────────────────────────────────
df_proc = df.copy()

# 1. Grupo de tenure (meses como cliente)
df_proc['tenure_group'] = pd.cut(
    df_proc['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['new_0_12', 'medium_12_24', 'loyal_24_48', 'champion_48_72'],
    include_lowest=True
)

# 2. Cargos por mes promedio (TotalCharges / tenure)
df_proc['avg_monthly_charge'] = np.where(
    df_proc['tenure'] > 0,
    df_proc['TotalCharges'] / df_proc['tenure'],
    df_proc['MonthlyCharges']
)

# 3. Número de servicios contratados
service_cols = ['PhoneService', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_proc['num_services'] = df_proc[service_cols].apply(
    lambda row: sum(v == 'Yes' for v in row), axis=1
)

# 4. Indicador: contrato mensual (alto riesgo según EDA)
df_proc['is_monthly_contract'] = (df_proc['Contract'] == 'Month-to-month').astype(int)

# 5. Indicador: cliente nuevo (tenure <= 6)
df_proc['is_new_customer'] = (df_proc['tenure'] <= 6).astype(int)

# 6. Indicador: fiber optic (alto riesgo según EDA)
df_proc['is_fiber_optic'] = (df_proc['InternetService'] == 'Fiber optic').astype(int)

# 7. Sin servicios de valor agregado
df_proc['no_value_added'] = (
    (df_proc['OnlineSecurity'] == 'No') &
    (df_proc['TechSupport'] == 'No') &
    (df_proc['OnlineBackup'] == 'No')
).astype(int)

print(f'✅ Features derivadas creadas. Shape: {df_proc.shape}')
new_feats = ['tenure_group', 'avg_monthly_charge', 'num_services', 
             'is_monthly_contract', 'is_new_customer', 'is_fiber_optic', 'no_value_added']
print('\n📋 Nuevas features:')
print(df_proc[new_feats].head(5))

✅ Features derivadas creadas. Shape: (7043, 27)

📋 Nuevas features:
  tenure_group  avg_monthly_charge  num_services  is_monthly_contract  \
0     new_0_12           29.850000             1                    1   
1  loyal_24_48           55.573529             3                    0   
2     new_0_12           54.075000             3                    1   
3  loyal_24_48           40.905556             3                    0   
4     new_0_12           75.825000             1                    1   

   is_new_customer  is_fiber_optic  no_value_added  
0                1               0               0  
1                0               0               0  
2                1               0               0  
3                0               0               0  
4                1               1               1  


## 3. Codificación de Variables Categóricas

In [4]:
# ─────────────────────────────────────────
# TARGET: Churn → 0/1
# ─────────────────────────────────────────
df_proc['Churn_enc'] = (df_proc['Churn'] == 'Yes').astype(int)
print(f'📊 Churn encoding: No=0, Yes=1')
print(df_proc['Churn_enc'].value_counts())

# ─────────────────────────────────────────
# VARIABLES BINARIAS SIMPLES (Yes/No → 1/0)
# ─────────────────────────────────────────
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df_proc[col + '_enc'] = (df_proc[col] == 'Yes').astype(int)

# SeniorCitizen ya es 0/1
# gender: Male=1, Female=0
df_proc['gender_enc'] = (df_proc['gender'] == 'Male').astype(int)

print('\n✅ Variables binarias codificadas')

# ─────────────────────────────────────────
# VARIABLES CON 3 VALORES: Yes / No / No internet service → 1/0/0
# Se tratan como binarias (No internet = No)
# ─────────────────────────────────────────
three_val_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                  'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in three_val_cols:
    df_proc[col + '_enc'] = (df_proc[col] == 'Yes').astype(int)

print('✅ Variables ternarias codificadas (Yes=1, No/No service=0)')

# ─────────────────────────────────────────
# ONE-HOT ENCODING: Contract, InternetService, PaymentMethod, tenure_group
# ─────────────────────────────────────────
ohe_cols = ['Contract', 'InternetService', 'PaymentMethod', 'tenure_group']

df_encoded = pd.get_dummies(df_proc, columns=ohe_cols, drop_first=False, dtype=int)

# Eliminar columnas originales ya procesadas
cols_to_drop = (['Churn', 'gender', 'Partner', 'Dependents',
                 'PhoneService', 'PaperlessBilling'] + three_val_cols)
df_encoded = df_encoded.drop(columns=cols_to_drop, errors='ignore')

print(f'\n✅ Shape tras codificación: {df_encoded.shape}')
print('\n📋 Columnas finales:')
for c in df_encoded.columns:
    print(f'  {c}')

📊 Churn encoding: No=0, Yes=1
Churn_enc
0    5174
1    1869
Name: count, dtype: int64

✅ Variables binarias codificadas
✅ Variables ternarias codificadas (Yes=1, No/No service=0)

✅ Shape tras codificación: (7043, 37)

📋 Columnas finales:
  SeniorCitizen
  tenure
  MonthlyCharges
  TotalCharges
  avg_monthly_charge
  num_services
  is_monthly_contract
  is_new_customer
  is_fiber_optic
  no_value_added
  Churn_enc
  Partner_enc
  Dependents_enc
  PhoneService_enc
  PaperlessBilling_enc
  gender_enc
  MultipleLines_enc
  OnlineSecurity_enc
  OnlineBackup_enc
  DeviceProtection_enc
  TechSupport_enc
  StreamingTV_enc
  StreamingMovies_enc
  Contract_Month-to-month
  Contract_One year
  Contract_Two year
  InternetService_DSL
  InternetService_Fiber optic
  InternetService_No
  PaymentMethod_Bank transfer (automatic)
  PaymentMethod_Credit card (automatic)
  PaymentMethod_Electronic check
  PaymentMethod_Mailed check
  tenure_group_new_0_12
  tenure_group_medium_12_24
  tenure_group_loyal

## 4. Split Train/Test Estratificado

In [5]:
# ─────────────────────────────────────────
# SPLIT 80/20 ESTRATIFICADO POR Churn
# ─────────────────────────────────────────
TARGET = 'Churn_enc'
FEATURES = [c for c in df_encoded.columns if c != TARGET]

X = df_encoded[FEATURES]
y = df_encoded[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y     # ← Preserva el ratio de clases en train y test
)

print(f'📦 Train: {X_train.shape[0]} registros | Test: {X_test.shape[0]} registros')
print(f'🎯 Features: {len(FEATURES)}')
print(f'\n📊 Distribución de clases:')
print(f'  Train → Churn: {y_train.mean()*100:.1f}% | No Churn: {(1-y_train.mean())*100:.1f}%')
print(f'  Test  → Churn: {y_test.mean()*100:.1f}%  | No Churn: {(1-y_test.mean())*100:.1f}%')
print('✅ Estratificación correcta: ratio preservado en ambos sets')

📦 Train: 5634 registros | Test: 1409 registros
🎯 Features: 36

📊 Distribución de clases:
  Train → Churn: 26.5% | No Churn: 73.5%
  Test  → Churn: 26.5%  | No Churn: 73.5%
✅ Estratificación correcta: ratio preservado en ambos sets


## 5. Escalamiento

In [6]:
# ─────────────────────────────────────────
# STANDARD SCALER — Ajustar solo en train
# ─────────────────────────────────────────
num_cols_to_scale = ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_charge', 'num_services']
num_cols_to_scale = [c for c in num_cols_to_scale if c in X_train.columns]

scaler_churn = StandardScaler()

X_train_sc = X_train.copy()
X_test_sc  = X_test.copy()

X_train_sc[num_cols_to_scale] = scaler_churn.fit_transform(X_train[num_cols_to_scale])
X_test_sc[num_cols_to_scale]  = scaler_churn.transform(X_test[num_cols_to_scale])

print('✅ Escalamiento aplicado')
print(f'   Columnas escaladas: {num_cols_to_scale}')
print('\n📊 Verificación (medias en train escalado):')
print(X_train_sc[num_cols_to_scale].describe().round(4))

✅ Escalamiento aplicado
   Columnas escaladas: ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_charge', 'num_services']

📊 Verificación (medias en train escalado):
          tenure  MonthlyCharges  TotalCharges  avg_monthly_charge  \
count  5634.0000       5634.0000     5634.0000           5634.0000   
mean     -0.0000         -0.0000        0.0000             -0.0000   
std       1.0001          1.0001        1.0001              1.0001   
min      -1.3223         -1.5440       -1.0089             -1.6911   
25%      -0.9560         -0.9712       -0.8321             -0.9560   
50%      -0.1419          0.1848       -0.3968              0.1870   
75%       0.9165          0.8319        0.6742              0.8446   
max       1.6085          1.7859        2.8019              1.8682   

       num_services  
count     5634.0000  
mean        -0.0000  
std          1.0001  
min         -1.5997  
25%         -1.0590  
50%          0.0223  
75%          0.5629  
max          2.1849

## 6. Guardado de Artefactos

In [7]:
import os
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Guardar datasets
X_train.to_csv('../data/processed/churn_X_train.csv', index=False)
X_test.to_csv('../data/processed/churn_X_test.csv', index=False)
X_train_sc.to_csv('../data/processed/churn_X_train_scaled.csv', index=False)
X_test_sc.to_csv('../data/processed/churn_X_test_scaled.csv', index=False)
y_train.to_csv('../data/processed/churn_y_train.csv', index=False)
y_test.to_csv('../data/processed/churn_y_test.csv', index=False)
df_encoded.to_csv('../data/processed/churn_processed.csv', index=False)

# Metadata
churn_feature_info = {
    'all_features': list(FEATURES),
    'numeric_features': num_cols_to_scale,
    'target': TARGET,
    'n_features': len(FEATURES),
    'n_train': len(X_train),
    'n_test': len(X_test),
    'churn_rate_train': float(y_train.mean())
}
with open('../data/processed/churn_feature_info.json', 'w') as f:
    json.dump(churn_feature_info, f, indent=2)

# Scaler
joblib.dump(scaler_churn, '../models/churn_scaler.pkl')

print('✅ Artefactos de Churn guardados:')
print('  📁 data/processed/churn_X_train.csv / churn_X_test.csv')
print('  📁 data/processed/churn_X_train_scaled.csv / churn_X_test_scaled.csv')
print('  📁 data/processed/churn_y_train.csv / churn_y_test.csv')
print('  📁 data/processed/churn_processed.csv')
print('  📁 data/processed/churn_feature_info.json')
print('  📁 models/churn_scaler.pkl')

✅ Artefactos de Churn guardados:
  📁 data/processed/churn_X_train.csv / churn_X_test.csv
  📁 data/processed/churn_X_train_scaled.csv / churn_X_test_scaled.csv
  📁 data/processed/churn_y_train.csv / churn_y_test.csv
  📁 data/processed/churn_processed.csv
  📁 data/processed/churn_feature_info.json
  📁 models/churn_scaler.pkl


## 7. 📝 Conclusiones

1. **TotalCharges:** 11 nulos imputados con 0 (clientes nuevos con tenure=0)
2. **Variables binarias:** Codificadas directamente Yes/No → 1/0
3. **Variables ternarias** (Yes/No/No internet service): Tratadas como binarias (Yes=1, resto=0)
4. **OHE:** Contract (3), InternetService (3), PaymentMethod (4), tenure_group (4)
5. **Split estratificado:** Preserva el 26.5% de Churn=Yes en ambos sets
6. **Features derivadas:** `num_services`, `is_monthly_contract`, `no_value_added` enriquecen el dataset
7. **StandardScaler** aplicado solo a numéricas continuas (fit solo en train)